# Weighted S10000 K-Tilde / Lambda Comparison

This notebook audits the four S10000 source laws used by the weighted experiment suite. It converts stored Fourier energies to the unitary convention and computes Lambda tables with

$$\mu_{1/2}(i)=\tfrac12\widetilde\mu_{\mathrm{S10000}}(i)+\tfrac{1}{2n}.$$

The five-trial convergence study keeps raw K-tilde errors while applying $\zeta=1/2$ to its sampling-law and Lambda diagnostics. Figures are written under `results/weighted/ktilde/figures/`.


In [ ]:
from pathlib import Path
import sys

import importlib
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    helper_candidates = [
        candidate / 'sd15_conditioning_experiment.py',
        candidate / 'sd1.5' / 'analyze_results' / 'sd15_conditioning_experiment.py',
    ]
    for helper_path in helper_candidates:
        if helper_path.is_file():
            helper_dir = helper_path.parent
            if str(helper_dir) not in sys.path:
                sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise FileNotFoundError('Could not find sd15_conditioning_experiment.py from the notebook cwd.')

import sd15_conditioning_experiment as exp
exp = importlib.reload(exp)

SD15_ROOT = exp.find_sd15_root(NOTEBOOK_DIR)
S10000_CONFIG = SD15_ROOT / 'ktilde' / 'weighted' / 'config_convergence.json'
ZETA = 0.5
CATALOG = exp.load_ktilde_catalog(SD15_ROOT, config_path=S10000_CONFIG)
TABLES = exp.build_lambda_tables(
    SD15_ROOT,
    config_path=S10000_CONFIG,
    probability_regularization_zeta=ZETA,
    skip_missing=True,
)
BANK_SUMMARY = pd.DataFrame(
    [
        {
            'name': name,
            'role': value.get('role', name),
            'label': value.get('label', name),
            'christoffel_law': value.get('christoffel_law', ''),
            'prompt': value.get('prompt', ''),
            'prompt_bank': ', '.join(value.get('prompt_bank', [])),
            'artifact_exists': (SD15_ROOT / 'ktilde' / 'weighted' / f'{name}.npz').is_file(),
        }
        for name, value in CATALOG.items()
    ]
)


def style_plain_numbers(frame):
    return frame.style.format(lambda value: exp.format_plain_number(value))


display(BANK_SUMMARY)
if TABLES['missing_names']:
    print('Missing k-tilde artifacts:', TABLES['missing_names'])
FFT_ENERGY_SCALES = sorted({int(round(value)) for value in TABLES['fft_energy_scale'].values()})
display(
    Markdown(
        "**Weighted source-law audit.** "
        "Stored `K_tilde` artifacts were estimated with the unnormalized FFT, so the absolute lambda and kappa tables below divide Fourier energies by `H * W` "
        f"({', '.join(str(value) for value in FFT_ENERGY_SCALES)}) to report the unitary-FFT convention. "
        "Every sampling column uses zeta=1/2 regularization; this changes the absolute compatibility values."
    )
)
display(TABLES['kappa_df'].style.format({'kappa_hat': exp.format_plain_number}))
display(
    TABLES['lambda_df'].style.format(
        {
            'lambda_hat': exp.format_plain_number,
            'kappa_hat': exp.format_plain_number,
            'mismatch_penalty': exp.format_plain_number,
        }
    )
)
display(style_plain_numbers(TABLES['lambda_table']))
display(style_plain_numbers(TABLES['penalty_table']))
display(
    TABLES['matched_check'].style.format(
        {
            'lambda_hat': exp.format_plain_number,
            'kappa_hat': exp.format_plain_number,
            'abs_lambda_minus_kappa': exp.format_plain_number,
        }
    )
)
display(
    Markdown(
        r"**$\widetilde{\Lambda}'$ interpretation.** "
        r"The $\widetilde{\lambda}$ heatmap shows the absolute compatibility cost of using sampling law "
        r"$\widetilde{\mu}_{c_s}$ with Christoffel function $c_r$. "
        r"$\widetilde{\Lambda}'$ is the row-normalized mismatch factor "
        r"$\widetilde{\lambda}(c_r,c_r,c_s)/\widetilde{\kappa}(c_r)$, so values near 1 mean the sampling law is close to the matched baseline for that row, while larger values show how much extra penalty you pay from mismatch."
    )
)


## Five-Trial Algorithm 1 K-Tilde Convergence

The four saved S10000 artifacts remain fixed references. For every prompt, five
new S10000 estimates use disjoint latent-seed blocks and record relative
$\ell^2$ error, relative $\ell^\infty$ error, the regularized Lambda
max-ratio, and the regularized maximum log-probability ratio every 10
iterations. The probability-based metrics use $\zeta=1/2$.

Each curve is the arithmetic mean of the five trial values at that iteration.
Shading is the 95% Student-$t$ confidence interval computed on the original
metric scale; the mean and bounds are then displayed on the existing
logarithmic $y$-axis. The completion table remains visible while any of the 20
jobs is missing or incomplete.


In [ ]:
TRIAL_COMPLETION = exp.ktilde_convergence_trial_completion_table(SD15_ROOT)
CONVERGENCE_FIGURE_DIR = SD15_ROOT / 'results' / 'weighted' / 'ktilde' / 'figures'
CONVERGENCE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
COMPLETION_PATH = CONVERGENCE_FIGURE_DIR / 'ktilde_convergence_five_trial_completion.csv'
TRIAL_COMPLETION.to_csv(COMPLETION_PATH, index=False)
display(TRIAL_COMPLETION)

CONVERGENCE_FIGURE_PATHS = {}
if not TRIAL_COMPLETION['status'].eq('complete').all():
    print('The five-trial convergence figures are pending. Run scripts/weighted/ktilde_convergence/list_all.sh to inspect all 20 jobs.')
else:
    CONVERGENCE_TRACES = exp.load_ktilde_convergence_trial_traces(SD15_ROOT)
    CONVERGENCE_SUMMARIES = exp.summarize_ktilde_convergence_trials(
        CONVERGENCE_TRACES,
        confidence_level=0.95,
    )
    CONVERGENCE_FIGURE_PATHS = exp.export_ktilde_convergence_trial_figure_set(
        CONVERGENCE_SUMMARIES,
        output_dir=CONVERGENCE_FIGURE_DIR,
        file_format='pdf',
        metrics=list(exp.KTILDE_TRIAL_CONVERGENCE_METRICS),
        show=True,
    )
    display(pd.Series({key: str(value) for key, value in CONVERGENCE_FIGURE_PATHS.items()}))
CONVERGENCE_FIGURE_PATHS


## Export Lambda Figures

This cell exports the absolute Lambda heatmap, the individual sampling-law
plots, and the compact sampling-law row. The displayed `FIGURE_PATHS` series
is the checklist of generated paper figures.


In [ ]:
FIGURE_DIR = SD15_ROOT / 'results' / 'weighted' / 'ktilde' / 'figures'
FIGURE_PATHS = exp.export_lambda_figure_set(
    TABLES,
    output_dir=FIGURE_DIR,
    file_format='pdf',
    show=True,
)
display(pd.Series({key: str(value) for key, value in FIGURE_PATHS.items()}))
FIGURE_PATHS


## Regularized S10000 Probability Audit

This audit records both the raw artifact statistics and the exact $\zeta=1/2$ law used by reconstruction and Lambda analysis.


In [ ]:
import numpy as np

probability_rows = []
for role, info in TABLES['bank'].items():
    raw = np.asarray(info['raw_probabilities'], dtype=np.float64).reshape(-1)
    effective = np.asarray(info['probabilities'], dtype=np.float64).reshape(-1)
    expected = 0.5 * raw + 0.5 / raw.size
    np.testing.assert_allclose(effective, expected, rtol=0.0, atol=2e-16)
    probability_rows.append(
        {
            'role': role,
            'artifact': info['name'],
            'n': raw.size,
            'zeta': info['probability_regularization_zeta'],
            'raw_sum': raw.sum(),
            'raw_min': raw.min(),
            'raw_max': raw.max(),
            'regularized_sum': effective.sum(),
            'regularized_min': effective.min(),
            'regularized_max': effective.max(),
            'required_floor': 0.5 / raw.size,
            'floor_satisfied': bool(effective.min() >= 0.5 / raw.size),
        }
    )
PROBABILITY_AUDIT = pd.DataFrame(probability_rows)
AUDIT_PATH = SD15_ROOT / 'results' / 'weighted' / 'ktilde' / 'figures' / 'probability_audit.csv'
AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
PROBABILITY_AUDIT.to_csv(AUDIT_PATH, index=False)
display(PROBABILITY_AUDIT)
AUDIT_PATH


## Sampling-CFG Ablation

This notebook compares empirical Christoffel sampling laws constructed at
sampling-side CFG values $1$, $3$, $5$, and $7.5$. It loads completed
artifacts incrementally; unfinished cells are shown explicitly. Set `ZETA` in
the setup cell to switch between the raw law and uniform regularization.


In [ ]:
from pathlib import Path
import importlib.util
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd / 'analyze_results/weighted/ktilde']
candidates.extend(parent / 'analyze_results/weighted/ktilde' for parent in (cwd, *cwd.parents))
HELPER_ROOT = next((path for path in candidates if (path / 'cfg_analysis.py').is_file()), None)
if HELPER_ROOT is None:
    raise FileNotFoundError('Could not locate weighted/ktilde/cfg_analysis.py.')
module_path = HELPER_ROOT / 'cfg_analysis.py'
module_spec = importlib.util.spec_from_file_location('weighted_cfg_ktilde_analysis', module_path)
cfg_analysis = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(cfg_analysis)
FIGURE_ROOT = cfg_analysis.FIGURE_ROOT
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

# Use 0.0 for the raw empirical law or 0.5 for the theory-aligned regularized law
ZETA = 0.5
ZETA_TAG = str(ZETA).replace('.', 'p')
print(f'Probability regularization: zeta={ZETA:g}')


## Completion and probability audit


In [ ]:
BANK, COMPLETION = cfg_analysis.load_bank(zeta=ZETA)
display(COMPLETION)
print(f'Complete artifacts: {len(BANK)} / 12')


## Regularized sampling distributions


In [ ]:
DISTRIBUTION_OUTPUT = cfg_analysis.plot_distribution_grid(
    BANK,
    output_path=FIGURE_ROOT / f'ktilde_cfg_sampling_distributions_zeta_{ZETA_TAG}.pdf',
    show=True,
)
DISTRIBUTION_OUTPUT


## Relative $\ell^2$ difference from CFG 7.5

This diagnostic compares the stored raw K-tilde estimates and is therefore
independent of the `ZETA` toggle.


In [ ]:
RELATIVE_L2 = cfg_analysis.relative_l2_table(BANK)
display(RELATIVE_L2)


## Absolute $\widetilde{\Lambda}$ compatibility


In [ ]:
LAMBDA_VALUES = cfg_analysis.lambda_table(BANK)
display(LAMBDA_VALUES)
LAMBDA_OUTPUT = cfg_analysis.plot_lambda_panels(
    LAMBDA_VALUES,
    output_path=FIGURE_ROOT / f'ktilde_cfg_lambda_panels_zeta_{ZETA_TAG}.pdf',
    show=True,
)
LAMBDA_OUTPUT


## Cross-Class and Self-Difference Estimates

This notebook visualizes the three ordered cross-class estimates
$\widetilde K(\mathbb F_{c}-\mathbb F_{c_{\mathrm{sb}}})$, where
$c\in\{c_{\mathrm{ca}},c_{\mathrm{uc}},c_{\mathrm{db}}\}$. Sunset beach is
always used to generate the second image in each secant. It also includes the
self-difference special case
$\widetilde K(\mathbb F_{c_{\mathrm{sb}}}-\mathbb F_{c_{\mathrm{sb}}})$.
Set `ZETA` below to view either the raw normalized law or a
uniform-regularized law.


In [ ]:
from pathlib import Path
import importlib.util
from IPython.display import display

cwd = Path.cwd().resolve()
study_relpath = Path('analyze_results/weighted/ktilde')
candidates = [cwd, cwd / study_relpath]
candidates.extend(parent / study_relpath for parent in cwd.parents)
HELPER_ROOT = next((path for path in candidates if (path / 'cross_cross_analysis.py.).is_file()), None)
if HELPER_ROOT is None:
    raise FileNotFoundError(f'Could not locate {study_relpath}/cross_analysis.py.')
module_path = HELPER_ROOT / 'cross_cross_analysis.py.
module_spec = importlib.util.spec_from_file_location('weighted_cross_class_analysis', module_path)
cross_analysis = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(cross_analysis)
FIGURE_ROOT = cross_analysis.FIGURE_ROOT
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

# Use 0.0 for the raw empirical law or 0.5 for uniform regularization
ZETA = 0.5
ZETA_TAG = str(ZETA).replace('.', 'p')


## Completion and probability audit


In [ ]:
BANK, COMPLETION = cross_analysis.load_bank(zeta=ZETA)
display(COMPLETION)
print(f'Complete artifacts: {len(BANK)} / 4')
print('Three proper cross-class estimates plus one sunset/sunset self-difference are expected.')


## Ordered cross-class sampling laws


In [ ]:
CROSS_CLASS_OUTPUT = cross_analysis.plot_cross_class_laws(
    BANK,
    output_path=FIGURE_ROOT / f'ktilde_cross_class_sampling_laws_zeta_{ZETA_TAG}.pdf',
    show=True,
)
CROSS_CLASS_OUTPUT


## Cross/self prompt compatibility

For each row, the numerator is one of the three cross-class estimates against
sunset beach or the sunset/sunset self-difference. The columns are the four
self-class Christoffel sampling laws. All columns use the selected `ZETA`.
Uniform MCS and inverse-square are intentionally excluded from every quantity
and statistic in this notebook.

The displayed quantity is the absolute unitary-scaled compatibility factor

$$
\widetilde\Lambda(c_r,c_s)
=
\max_i
\frac{\widetilde K(\mathbb F_{c_r}-\mathbb F_{c_{\mathrm{sb}}})(i)}
     {\widetilde\mu_{c_s}(i)}.
$$


In [ ]:
LAMBDA_VALUES = cross_analysis.compatibility_table(
    BANK,
    probability_regularization_zeta=ZETA,
)
LAMBDA_TABLE = LAMBDA_VALUES.pivot(
    index='reconstruction_condition',
    columns='sampling_condition',
    values='lambda_hat',
).reindex(
    index=cross_analysis.RECOVERY_ORDER,
    columns=cross_analysis.SAMPLING_ORDER,
)
display(LAMBDA_TABLE)


In [ ]:
LAMBDA_OUTPUT = cross_analysis.plot_lambda_matrix(
    LAMBDA_VALUES,
    output_path=FIGURE_ROOT / f'ktilde_cross_class_lambda_zeta_{ZETA_TAG}.pdf',
    show=True,
)
LAMBDA_OUTPUT


## Does cross/self compatibility predict reconstruction performance?

This secondary diagnostic uses only the four Christoffel sampling laws. For
each sampling/recovery combination, reconstruction metrics are averaged over
the available balanced observations and compared with
$\widetilde\Lambda/\widetilde\kappa$. The normalization permits recovery
prompts with differently scaled numerators to be pooled; it does not change
the sampling-law ordering within a fixed numerator row.

Spearman correlation compares ranks rather than raw magnitudes. The reported
agreement coefficient is sign-adjusted so that a positive value always means
that a larger compatibility penalty accompanies worse performance.


### Completed out-of-range experiment


In [ ]:
OOD_ROWS_ALL = cross_analysis.load_ood_rows()
OOD_ROWS = OOD_ROWS_ALL[
    OOD_ROWS_ALL['sampling_condition'].astype(str).isin(cross_analysis.SAMPLING_ORDER)
].copy()
print(f'Christoffel-only out-of-range rows: {len(OOD_ROWS)} / 400')
OOD_COMPARISON = cross_analysis.compatibility_performance_analysis(
    BANK,
    rows=OOD_ROWS,
    probability_regularization_zeta=ZETA,
)
display(
    OOD_COMPARISON['merged'][[
        'reconstruction_condition',
        'sampling_label',
        'lambda_hat',
        'kappa_hat',
        'mismatch_penalty',
        'bp_best_loss_mean',
        'psnr_db_mean',
        'ssim_mean',
        'lpips_mean',
        'pixel_mae_mean',
    ]].sort_values(['reconstruction_condition', 'mismatch_penalty'])
)
display(OOD_COMPARISON['best_match'])
display(OOD_COMPARISON['correlations'])


### Prompt-matched experiment in progress

To avoid favoring combinations that have progressed farther, this comparison
uses only sampling-ratio/trial pairs completed by all 16 Christoffel
sampling/recovery combinations. It updates automatically as the experiment
finishes.


In [ ]:
PROMPT_MATCHED_ROWS_ALL = cross_analysis.load_prompt_matched_rows()
PROMPT_MATCHED_ROWS = cross_analysis.balanced_prompt_matched_rows(PROMPT_MATCHED_ROWS_ALL)
print(f'All finalized prompt-matched rows: {len(PROMPT_MATCHED_ROWS_ALL)} / 400')
print(f'Rows in the balanced comparison: {len(PROMPT_MATCHED_ROWS)}')
if PROMPT_MATCHED_ROWS.empty:
    print('No rate/trial pair is complete across all 16 combinations yet.')
else:
    PROMPT_MATCHED_COMPARISON = cross_analysis.compatibility_performance_analysis(
        BANK,
        rows=PROMPT_MATCHED_ROWS,
        probability_regularization_zeta=ZETA,
    )
    display(PROMPT_MATCHED_COMPARISON['best_match'])
    display(PROMPT_MATCHED_COMPARISON['correlations'])
